# Introduction to SQL

**SQL (Structured Query Language)** is a language used to work with **databases**.

Databases store data in **tables**, which are organized into:
- **Rows** (individual records)
- **Columns** (attributes or variables)

SQL allows us to:
- Retrieve data
- Filter data
- Combine data from multiple tables
- Summarize data

In this course, we will use SQL to **extract data for analysis**.

## Connecting to a Database with SQLite

In Python, we can connect to a database using the **sqlite3** library.

SQLite is a lightweight database system that stores the database in a **single file**.

Typical steps:
1. Import the `sqlite3` library
2. Create a connection to the database file
3. Create a cursor to execute SQL queries

In [20]:
# import the sqlite3 library
import sqlite3

# create a connection to the database
con = sqlite3.connect("../data/movies.sqlite")

# create a cursor we can use to execute queries
cur = con.cursor()

## SELECT and FROM

In SQL, we use **SELECT** and **FROM** to retrieve data from a table.

- `SELECT` specifies the **columns** we want to return.
- `FROM` specifies the **table** the data comes from.

Basic structure:

SELECT column1, column2  
FROM table_name;

To select **all columns**, we can use `*`.

## WHERE

The **WHERE** clause is used to **filter rows** based on a condition.

It allows us to select only the rows that meet a specified condition.

Basic structure:

SELECT column1, column2  
FROM table_name  
WHERE condition;

In [56]:
res = cur.execute("""
    SELECT original_title FROM movies
    WHERE vote_average > 8
    LIMIT 10
""")

for row in res:
    print(row)

('The Dark Knight',)
('Interstellar',)
('Inception',)
('The Lord of the Rings: The Return of the King',)
('Fight Club',)
('The Green Mile',)
('Forrest Gump',)
('Se7en',)
('Once Upon a Time in America',)
("Schindler's List",)


In [40]:
res = cur.execute("""
    SELECT original_title FROM movies
    WHERE vote_average > 8
    AND vote_count > 10000
""")

for row in res:
    print(row)

('The Dark Knight',)
('Interstellar',)
('Inception',)


## GROUP BY

The **GROUP BY** clause is used to **group rows that have the same value** in one or more columns.

This is commonly used with **aggregation functions**, such as:
- `COUNT()`
- `SUM()`
- `AVG()`
- `MIN()`
- `MAX()`

Basic structure:

SELECT column, AGG_FUNCTION(column)  
FROM table_name  
GROUP BY column;

In [53]:
res = cur.execute("""
    SELECT gender, COUNT(gender) FROM directors
    GROUP BY gender
""")

res.fetchall()


[(0, 625), (1, 150), (2, 1574)]

## HAVING

The **HAVING** clause is used to **filter grouped results**.

While `WHERE` filters **rows before grouping**, `HAVING` filters **groups after `GROUP BY`**.

Basic structure:

SELECT column, AGG_FUNCTION(column)  
FROM table_name  
GROUP BY column  
HAVING condition;

In [57]:
res = cur.execute("""
    SELECT director_id, COUNT(director_id) AS movie_count FROM movies
    GROUP BY director_id
    HAVING movie_count > 5
    LIMIT 10
""")

for row in res:
    print(row)

(4762, 7)
(4763, 7)
(4764, 7)
(4765, 8)
(4767, 11)
(4771, 7)
(4772, 8)
(4773, 8)
(4776, 7)
(4777, 9)


## ORDER BY

The **ORDER BY** clause is used to **sort the results** of a query.

By default, results are sorted in **ascending order (`ASC`)**.

You can also sort in **descending order (`DESC`)**.

Basic structure:

SELECT column1, column2  
FROM table_name  
ORDER BY column;

In [59]:
res = cur.execute("""
    SELECT director_id, COUNT(director_id), SUM(revenue) AS total_revenue FROM movies
    GROUP BY director_id
    ORDER BY total_revenue DESC
    LIMIT 10
""")

for row in res:
    print(row)

(4799, 27, 9147393164)
(4777, 9, 6498642820)
(4762, 7, 5883569439)
(4788, 12, 5832524638)
(4765, 8, 4227483234)
(4885, 11, 3725631503)
(4804, 13, 3590622002)
(4901, 5, 3339113893)
(4785, 14, 3337418241)
(4779, 16, 3189557997)


## JOIN (LEFT JOIN)

A **JOIN** is used to combine data from **multiple tables**.

Tables are connected using a **shared key**, which is a column that appears in both tables.

A common type is a **LEFT JOIN**:

- Keeps **all rows from the left table**
- Adds matching rows from the right table
- If no match exists, the result contains `NULL`

Basic structure:

SELECT columns  
FROM table_A  
LEFT JOIN table_B  
ON table_A.key = table_B.key;

In [72]:
res = cur.execute("""
    SELECT * 
    FROM directors
    LEFT JOIN movies
    ON directors.id = movies.director_id
    LIMIT 3
""")

for row in res:
    print(row)

('James Cameron', 4762, 2, 2710, 'Directing', 46000, 'Aliens', 18500000, 67, '1986-07-18', 183316455, 'Aliens', 7.7, 3220, "When Ripley's lifepod is found by a salvage crew over 50 years later, she finds that terra-formers are on the very planet they found the alien species. When the company sends a family of colonists out to investigate her story, all contact is lost with the planet and colonists. They enlist Ripley and the colonial marines to return and search for answers.", "This Time It's War", 679, 4762)
('James Cameron', 4762, 2, 2710, 'Directing', 43597, 'Avatar', 237000000, 150, '2009-12-10', 2787965087, 'Avatar', 7.2, 11800, 'In the 22nd century, a paraplegic Marine is dispatched to the moon Pandora on a unique mission, but becomes torn between following orders and protecting an alien civilization.', 'Enter the World of Pandora.', 19995, 4762)
('James Cameron', 4762, 2, 2710, 'Directing', 43876, 'Terminator 2: Judgment Day', 100000000, 101, '1991-07-01', 520000000, 'Terminator

In [80]:
res = cur.execute("""
    SELECT directors.name, 
    COUNT(directors.name) AS movie_count,
    SUM(movies.revenue) AS total_revenue
    FROM directors
    LEFT JOIN movies
    ON directors.id = movies.director_id
    GROUP BY directors.name
    ORDER BY total_revenue DESC
    LIMIT 10
""")

for row in res:
    print(row)

('Steven Spielberg', 27, 9147393164)
('Peter Jackson', 9, 6498642820)
('James Cameron', 7, 5883569439)
('Michael Bay', 12, 5832524638)
('Christopher Nolan', 8, 4227483234)
('Chris Columbus', 11, 3725631503)
('Robert Zemeckis', 13, 3590622002)
('George Lucas', 5, 3339113893)
('Tim Burton', 14, 3337418241)
('Ridley Scott', 16, 3189557997)


## Types of JOINs

SQL provides several types of joins for combining tables.

**INNER JOIN**
- Returns only rows where there is a match in both tables.

**LEFT JOIN**
- Returns all rows from the left table.
- Includes matching rows from the right table.

**RIGHT JOIN**
- Returns all rows from the right table.
- Includes matching rows from the left table.

**FULL OUTER JOIN**
- Returns all rows from both tables.
- Non-matching rows contain `NULL` values.

In [81]:
res = cur.execute("""
    SELECT directors.name, 
    COUNT(directors.name) AS movie_count,
    SUM(movies.revenue) AS total_revenue
    FROM movies
    LEFT JOIN directors
    ON movies.director_id = directors.id
    GROUP BY directors.name
    ORDER BY total_revenue DESC
    LIMIT 10
""")

for row in res:
    print(row)

('Steven Spielberg', 27, 9147393164)
('Peter Jackson', 9, 6498642820)
('James Cameron', 7, 5883569439)
('Michael Bay', 12, 5832524638)
('Christopher Nolan', 8, 4227483234)
('Chris Columbus', 11, 3725631503)
('Robert Zemeckis', 13, 3590622002)
('George Lucas', 5, 3339113893)
('Tim Burton', 14, 3337418241)
('Ridley Scott', 16, 3189557997)


In [90]:
res = cur.execute("""
    SELECT * 
    FROM movies
    INNER JOIN directors
    ON movies.uid = directors.uid
    LIMIT 10
""")

for row in res:
    print(row)

(43602, 'Spider-Man 3', 258000000, 115, '2007-05-01', 890871626, 'Spider-Man 3', 5.9, 3576, 'The seemingly invincible Spider-Man goes up against an all-new crop of villain – including the shape-shifting Sandman. While Spider-Man’s superpowers are altered by an alien organism, his alter ego, Peter Parker, deals with nemesis Eddie Brock and also gets caught up in a love triangle.', 'The battle within.', 559, 4767, 'Wally Pfister', 4953, 2, 559, 'Directing')
(43655, 'Rush Hour 3', 140000000, 22, '2007-08-08', 258022233, 'Rush Hour 3', 6.1, 783, "After an attempted assassination on Ambassador Han, Inspector Lee and Detective Carter are back in action as they head to Paris to protect a French woman with knowledge of the Triads' secret leaders. Lee also holds secret meetings with a United Nations authority, but his personal struggles with a Chinese criminal mastermind named Kenji, which reveals that it's Lee's long-lost...brother.", 'The Rush Is On!', 5174, 4786, 'Barry Sonnenfeld', 4776, 2,